# Solve Field

This notebook can be used to determine the WCS solution of an OmegaCAM image.
It uses astrometry.net solve_field command. It is a necessary step before running scamp which needs a WCS accurate to approx 1", otherwise it fails.

Before start please be sure you can run the solve_field command from the astrometry package by tipying
/usr/local/astrometry/bin/solve-field

If you get something like
ERROR: You didn't specify any files to process.<br>
This program is part of the Astrometry.net suite.<br>
For details, visit http://astrometry.net.<br>
Git URL https://github.com/dstndstn/astrometry.net<br>
Revision 0.89, date Mon_Jan_17_07:47:26_2022_-0500.<br>

then you are okay.

This notebook will create a independent fits image for each CCD within the input mosaic. Then run solve_field on each one to find an astrometric solution. This solution will be incorporated to a new mosaic which will be use later.
Please be sure you have enough space in your hard disk, as this notebook will effectively created two new mosaic images for each input one.

Tested on my personal laptop using the index fields 4100 and 5200 series from astromatic.
Known issues:
- For some reason, the astrometry package failed during the flow of the script, but running the same command in terminal directly over the CCD fits image retrieves a solution in seconds.
- For those CCDs that have failed at retrieve a solution with astrometry, are copied directly into the final mosaic, causing some shifts between CCDs.

Lientur

In [1]:
import os
import glob
import logging
import multiprocessing

import pycavi_solve
print(pycavi_solve.__file__)

import pycavi_tools

/Users/fselman/Software/Pycavi/pycavi_solve.py


In [2]:
def run_py4_solve(observation_dir: str, cpu_count: int = 32, redo_astrometry: bool = False):
    """
    Run astrometry over a given observation directory.
    For each _pre_wcs fits image within the folder, it will create an independent fits image for each extension,
    for which later will be run astrometry net to create a wcs solution.
    Tested with the 4100 and 5200 astromatric indexes series.
    Be careful with the space, because many new fits images will be created.
    
    cpu_count: number of processors when calling the multiprocessing
    redo_astrometry: it will look if the final _wcs.fits mosaic exists. If redo_astrometry is False will not create
                     a new one. If True will created again an overwrite the previous one.
    
    create_wcs_frame accepts two different modes when dealing with failed CCDs: the continue mode in which it will
    copy the original, unsolved CCD into the new mosaic together with the rest of solved CCDs. The other available
    option is remove, in which the unsolved CCDs will not be included in the final mosaic, composed only by solved
    CCDs.
    """

    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s:%(levelname)s:%(message)s",
                        handlers=[
                            logging.FileHandler(observation_dir + "/pymeffpp_solve.log"),
                            logging.StreamHandler()])

    os.chdir(observation_dir)
    logging.info('current directory: {0}'.format(os.getcwd()))
    list_to_solve = glob.glob('OMEGA.*_pre.fits')

    if len(list_to_solve) == 0:
        logging.info("No images to solve, skipping directory")
        return

    i = 0
    for file_name in list_to_solve:
        i += 1
        logging.info("solving image {0:02d}/{1:02d}: {2}".format(i, len(list_to_solve), file_name[:-5] + '.fits', ))
        # if wcs image exist, continue to the next
        # this must be more flexible, allowing the option to redo the astrometry if the user wants to.
        mosaic_wcs = observation_dir + "/" + file_name[:-5] + "/" + file_name[:-5] + "_wcs.fits"
        if (os.path.isfile(mosaic_wcs) and not redo_astrometry):
            logging.info("wcs image already exist, skipping to the next image")
            continue

        # Solving
        logging.info("\n\n\nSolving field {0}".format(file_name[:-5]))
        pycavi_solve.run_solve_field(file_name[:-5], cpu_count=cpu_count)

        # Creating WCS file
        logging.info("Creating wcs mosaic file {0}\n\n\n".format(file_name))
        pycavi_solve.create_wcs_frame(os.path.abspath(file_name), mode="continue")

In [3]:
import multiprocessing

multiprocessing.cpu_count()

8

In [4]:
# Set the reduced science dir where it will look for the reduced science images, same as previous notebooks
# For each target within the folder will check the filters before start to work on each observation.
# In this case, during the tests I consider only the images in the filter NB_659.

config_dir = os.getcwd()+'/config/'
config = pycavi_tools.read_pycavi_config(config_dir)
reduced_science_dir = config['Directories']['reduced_science_dir']
log_dir = config['Directories']['project_logs_dir']

# a list with the tags of image filters. During the tests I worked only with r_SDSS and NB_659
filters_to_sol = ["r_SDSS"]

# target list will include all the observed targets within the reduced_science directory
target_list = []
for object_ in glob.glob(reduced_science_dir + "*"):
    target_list.append(object_)
target_list.sort()

# for each observed target will all the images which filters correspond to the ones given in filters_to_sol
for target_ in target_list:
    for filter_ in filters_to_sol:
        for observation_ in glob.glob(target_ + "/" + filter_ + "/*"):
                run_py4_solve(observation_, cpu_count=multiprocessing.cpu_count()-2, redo_astrometry=False)

# job done
print("Py4 Done!")

2024-06-19 15:38:52,342:INFO:current directory: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49
2024-06-19 15:38:52,343:INFO:solving image 01/10: OMEGA.2017-03-13T02:44:33.457_pre.fits
2024-06-19 15:38:52,344:INFO:


Solving field OMEGA.2017-03-13T02:44:33.457_pre
2024-06-19 15:38:52,394:INFO:run_solve_field: processing extensions...
2024-06-19 15:38:52,396:INFO:run_solve_field: number of extensions: 00
2024-06-19 15:38:52,479:INFO:solve_field: ------------ done ----------
2024-06-19 15:38:52,496:INFO:Creating wcs mosaic file OMEGA.2017-03-13T02:44:33.457_pre.fits



2024-06-19 15:38:52,497:INFO:create_wcs_frame: file_name: OMEGA.2017-03-13T02:44:33.457_pre.fits
2024-06-19 15:38:52,499:INFO:create_wcs_frame: file exists: OMEGA.2017-03-13T02:44:33.457_pre.fits
2024-06-19 15:38:52,500:INFO:create_wcs_frame: directory exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-1

wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__65_solve_field//ESO_CCD__65_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__66_solve_field//ESO_CCD__66_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__67_solve_field//ESO_CCD__67_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__68_solve_field//ESO_CCD__68_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__69_solve_field//ESO_CCD__69_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/C

2024-06-19 15:38:52,715:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__81_solve_field//ESO_CCD__81.wcs
2024-06-19 15:38:52,730:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__82_solve_field//ESO_CCD__82.wcs
2024-06-19 15:38:52,743:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__83_solve_field//ESO_CCD__83.wcs
2024-06-19 15:38:52,755:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__84_solve_field//ESO_CCD__84.wcs
2024-06-19 15:38:52,768:INFO:create_wcs_frame: W

wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__81_solve_field//ESO_CCD__81_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__82_solve_field//ESO_CCD__82_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__83_solve_field//ESO_CCD__83_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__84_solve_field//ESO_CCD__84_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__85_solve_field//ESO_CCD__85_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/C

2024-06-19 15:38:52,923:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__96_solve_field//ESO_CCD__96.wcs
2024-06-19 15:38:52,935:INFO:create_wcs_frame: creating /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/OMEGA.2017-03-13T02:44:33.457_pre_wcs.fits


...

Field: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__92_solve_field//ESO_CCD__92.fits
Field center: (RA,Dec) = (122.905491, -34.905150) deg.
Field center: (RA H:M:S, Dec D:M:S) = (08:11:37.318, -34:54:18.539).
Field size: 7.26427 x 14.5389 arcminutes
Field rotation angle: up is -0.108004 degrees E of N
Field parity: neg
Creating new FITS file "/Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__92_solve_field//ESO_CCD__92_newwcs.fits"...




wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__65_solve_field//ESO_CCD__65_newwcs.fits


2024-06-19 15:49:05,719:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__66_solve_field//ESO_CCD__66.wcs
2024-06-19 15:49:05,734:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__67_solve_field//ESO_CCD__67.wcs
2024-06-19 15:49:05,747:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__68_solve_field//ESO_CCD__68.wcs
2024-06-19 15:49:05,767:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__69_solve_field//ESO_CCD__69.wcs
2024-06-19 15:49:05,781:INFO:create_wcs_frame: W

wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__66_solve_field//ESO_CCD__66_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__67_solve_field//ESO_CCD__67_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__68_solve_field//ESO_CCD__68_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__69_solve_field//ESO_CCD__69_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__70_solve_field//ESO_CCD__70_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/C

2024-06-19 15:49:05,983:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__83_solve_field//ESO_CCD__83.wcs
2024-06-19 15:49:05,995:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__84_solve_field//ESO_CCD__84.wcs
2024-06-19 15:49:06,005:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__85_solve_field//ESO_CCD__85.wcs
2024-06-19 15:49:06,015:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__86_solve_field//ESO_CCD__86.wcs
2024-06-19 15:49:06,024:INFO:create_wcs_frame: W

wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__83_solve_field//ESO_CCD__83_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__84_solve_field//ESO_CCD__84_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__85_solve_field//ESO_CCD__85_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__86_solve_field//ESO_CCD__86_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__87_solve_field//ESO_CCD__87_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/C